## **前言**

*本次代码并没有直接运行，因为在检查代码的过程中发现，当前各类型数据的处理方式并不好，每种数据源都设计了单独的提取和切分策略，且 pdf 的提取工具 pymupdf 会因为双栏问题造成提取质量糟糕，因此计划做一次充分的调研再来设计整体的架构，计划先调研 Gemini notebook、ima 的数据提取和转换方式，看是否需要采取Markdown 作为中间格式的策略：【 任意来源 → 转换为 Markdown → 统一的 chunk_markdown → 向量库】，并且，如果针对双栏 pdf 提取的效果比较难，没有合适的工具的话，打算将论文（普遍双栏）的提取功能置后，先用 pymupdf ，完成核心的问答链路后，再做论文的处理*

### **针对 V1.1 实验结果分析和改进**

**上一版实验暴露的问题总结**
| 问题 | 严重程度 | 本版修复方式 |
|------|---------|-------------|
| 超长 chunk（最大 6340 字符），embedding 模型会截断 | 严重 | 新增句级二次切分兜底，单 chunk 硬上限 1000 字符 |
| 极短 chunk（最小 18 字符），噪声进入向量库 | 中等 | 空标题/极短段落统一过滤 |
| clean_pdf_text 过滤力度未验证，可能误删正文 | 中等 | 新增 verbose 诊断输出，可见被删行 |
| 网页被 Anubis 反爬拦截，内容无效 | 中等 | 改为用户提供 URL，换用无反爬保护的页面 |
| GitHub/Gitee 样本固定，质量难以保证 | 低 | 改为用户提供仓库链接，API 自动获取所有 .md 文件 |

**六类数据源获取和处理方式更新**
| 编号 | 来源类型 | 样本 | 获取方式 |
|------|---------|------|----------|
| 1 | 论文 PDF | 机器人运动控制相关论文 | 手动上传 PDF |
| 2 | 规则 PDF | 比赛规则 / 技术规范 | 手动上传 PDF |
| 3 | arXiv 论文 | 搜索关键词自动获取 | arXiv API（自动） |
| 4 | GitHub 项目 | 用户提供仓库链接，自动获取所有 .md | GitHub API（链接触发） |
| 5 | Gitee 项目 | 用户提供仓库链接，自动获取所有 .md | Gitee API（链接触发） |
| 6 | 普通网页 | 用户提供 URL，自动抓取正文 | HTTP + HTML 解析 |

### **切分函数 & 校验函数 设计**

**切分策略**

第一层：

段落感知切分
按「连续两个以上换行 = 空行 = 段落边界」切段落，累积到接近 `target_chars`（默认 500 字）才切一刀，不从段落中间截断，保证每个 chunk 语义完整。

第二层：

句级二次切分（新增）
若单个段落本身超过 `max_chars`（默认 1000 字），则对该段落进一步按中英文句末标点（。！？.!?）切句，累积到 `target_chars` 后提交。
目的：避免超长段落作为单个 chunk 被 embedding 模型截断（如 text-embedding-3-small 上限 8191 token）。

> 极短 chunk 过滤（新增）

低于 `min_chars`（默认 80 字）的缓冲区视为碎片：有前一个 chunk 时合并进去，否则直接丢弃。
目的：空标题、孤立页码等噪声碎片不进入向量库。

> Markdown 专用切分

对 .md 文件优先按 1~3 级标题（`#` / `##` / `###`）切 section，section 内容过短则过滤，超长则调用段落切分细切。

**校验策略**

字符偏移函数的作用：每个 chunk 记录 `char_start` 和 `char_end`，即它在原始文本中的起止位置（字符数）。
用户点击「查看来源」时，系统用这两个数字定位并高亮对应段落。
偏移记错会导致高亮错位，所以切分后必须立即用 `validate_chunks()` 校验。


In [3]:
import re

# ----- 切分 --------------------------------------------------------------------------------

# 问题：直接按行切分会把整页密集正文合并成一个超长段落，丢失语义边界
# 解决：按「连续两个以上换行（空行）」切段落，end 延伸到下一段落起始，把段间空白纳入当前段落范围，避免字符偏移出现缺口
# 目的：提供准确的段落起止偏移，作为 chunk_text 的输入
def split_paragraphs(text):
    raw_segments = list(re.finditer(r'(?:(?!\n\n).)+', text, re.DOTALL))
    paragraphs = []
    for i, m in enumerate(raw_segments):
        content = m.group().strip()
        if not content:
            continue
        p_start = m.start()
        p_end = raw_segments[i + 1].start() if i + 1 < len(raw_segments) else len(text)
        paragraphs.append((p_start, p_end, content))
    return paragraphs

# 问题：单个段落有时超过 1000 字符，整段作为一个 chunk 会被 embedding 模型截断
# 解决：按中英文句末标点（。！？.!?）将段落切成句子，保留标点在句尾
# 目的：为 flush_or_split 提供句子粒度的切分，确保 chunk 不超过 max_chars
def split_sentences(text):
    sentences = []
    pattern = re.compile(r'[^。！？.!?]*[。！？.!?]+')
    last_end = 0
    for m in pattern.finditer(text):
        sentences.append((m.start(), m.end(), m.group()))
        last_end = m.end()
    if last_end < len(text) and text[last_end:].strip():
        sentences.append((last_end, len(text), text[last_end:]))
    return sentences

# 问题1：段落本身超长时没有兜底，整段直接输出为一个超大 chunk
# 问题2：极短段落（空标题、孤立标点行）进入向量库，污染检索结果
# 解决：新增 max_chars 参数，超过上限触发句级二次切分；flush 阶段过滤 < min_chars 的碎片，有前一 chunk 则合并，否则丢弃
# 目的：所有 chunk 长度落在 [min_chars, max_chars] 范围内，适合 embedding
def chunk_text(text, doc_id, doc_type, target_chars=500, min_chars=80, max_chars=1000):
    """
    参数：
      target_chars: 每个 chunk 的目标字符数
      min_chars:    低于此值的缓冲区视为碎片
      max_chars:    单个 chunk 的硬上限，超过则触发句级二次切分
    """
    paragraphs = split_paragraphs(text)
    chunks = []
    buf_start = buf_end = None
    chunk_index = 0

    def flush(b_start, b_end):
        nonlocal chunk_index
        if b_start is None:
            return
        chunk_str = text[b_start:b_end]
        if not chunk_str.strip():
            return
        if len(chunk_str.strip()) < min_chars and chunks:
            prev = chunks[-1]
            prev['char_end'] = b_end
            prev['chunk_text'] = text[prev['char_start']:b_end]
            prev['char_len'] = len(prev['chunk_text'])
            return
        if len(chunk_str.strip()) < min_chars and not chunks:
            return  # 首个 chunk 也太短（如空标题），丢弃
        chunks.append({
            'doc_id': doc_id, 'doc_type': doc_type, 'chunk_index': chunk_index,
            'char_start': b_start, 'char_end': b_end,
            'chunk_text': chunk_str, 'char_len': len(chunk_str),
        })
        chunk_index += 1

    def flush_or_split(b_start, b_end):
        """提交前检查是否超过 max_chars，超过则先按句子切分再提交。"""
        nonlocal chunk_index
        if b_start is None:
            return
        chunk_str = text[b_start:b_end]
        if not chunk_str.strip():
            return
        if len(chunk_str) <= max_chars:
            flush(b_start, b_end)
            return
        sentences = split_sentences(chunk_str)
        if not sentences:
            flush(b_start, b_end)
            return
        sub_buf_start = sub_buf_end = None
        for s_start, s_end, _ in sentences:
            abs_s_start = b_start + s_start
            abs_s_end = b_start + s_end
            if sub_buf_start is None:
                sub_buf_start, sub_buf_end = abs_s_start, abs_s_end
                continue
            if (sub_buf_end - sub_buf_start) >= target_chars:
                flush(sub_buf_start, sub_buf_end)
                sub_buf_start, sub_buf_end = abs_s_start, abs_s_end
            else:
                sub_buf_end = abs_s_end
        flush(sub_buf_start, sub_buf_end)

    for p_start, p_end, _seg in paragraphs:
        if buf_start is None:
            buf_start, buf_end = p_start, p_end
            continue
        if (buf_end - buf_start) >= target_chars:
            flush_or_split(buf_start, buf_end)
            buf_start, buf_end = p_start, p_end
        else:
            buf_end = p_end
    flush_or_split(buf_start, buf_end)
    return chunks

# 问题：对 .md 文件用纯字符长度切分，会割断标题章节语义，空标题节（只有一行 ## 标题、无正文）也被切出独立 chunk，产生极短噪声
# 解决：优先按 1~3 级标题切 section；过滤正文 < min_chars 的 section；超长 section 调用 chunk_text 细切
# 目的：每个 chunk 对应完整章节，消除空标题噪声
def chunk_markdown(text, doc_id, doc_type, target_chars=500, min_chars=80, max_chars=1000):
    sections = re.split(r'(?m)^(?=#{1,3} )', text)
    all_chunks = []
    search_start = 0
    for section in sections:
        if not section.strip():
            search_start += len(section)
            continue
        lines = section.splitlines()
        body = '\n'.join(lines[1:]).strip() if len(lines) > 1 else ''
        if len(body) < min_chars and len(section.strip()) < min_chars:
            search_start += len(section)
            continue
        sec_start = text.find(section, search_start)
        sec_end = sec_start + len(section)
        search_start = sec_end
        if len(section) <= max_chars:
            all_chunks.append({
                'doc_id': doc_id, 'doc_type': doc_type,
                'chunk_index': len(all_chunks),
                'char_start': sec_start, 'char_end': sec_end,
                'chunk_text': section, 'char_len': len(section),
            })
        else:
            sub_chunks = chunk_text(section, doc_id=doc_id, doc_type=doc_type,
                                    target_chars=target_chars, min_chars=min_chars,
                                    max_chars=max_chars)
            for sc in sub_chunks:
                sc['char_start'] += sec_start
                sc['char_end'] += sec_start
                sc['chunk_text'] = text[sc['char_start']:sc['char_end']]
                sc['char_len'] = len(sc['chunk_text'])
                sc['chunk_index'] = len(all_chunks)
                all_chunks.append(sc)
    return all_chunks


# ----- 验证 --------------------------------------------------------------------------------

# 目的：用 char_start/char_end 从原始文本重新切片，与存储的 chunk_text 逐字比对，pass=True 才能进入 Embedding，否则说明切分偏移有 bug
def validate_chunks(text, chunks):
    mismatches = []
    for c in chunks:
        rebuilt = text[c['char_start']:c['char_end']]
        if rebuilt != c['chunk_text']:
            mismatches.append({
                'chunk_index': c['chunk_index'],
                'stored_preview': c['chunk_text'][:50],
                'rebuilt_preview': rebuilt[:50],
            })
    return {
        'total_chunks': len(chunks),
        'mismatch_count': len(mismatches),
        'mismatches': mismatches,
        'pass': len(mismatches) == 0,
    }


print('共用函数定义完成，后续模块可调用')


共用函数定义完成，后续模块可调用


### **数据类型1：论文PDF**

**1. 上个版本遇到的问题**
- **超长 chunk**：论文 PDF 段落密集，部分段落超过 1000 字符，原本按照段落切分直接作为 chunk 会被 embedding 模型截断，导致语义丢失
- **噪声污染**：pymupdf 提取的原始文本含页眉页脚、孤立页码、参考文献列表，会降低向量质量
- **过滤力度未验证**：原本`clean_pdf_text` 的「< 8 字符行一律丢弃」规则可能误删公式行、短节标题（如 `A.`）

**2. 本次实验采取的解决方式**
- `chunk_text()` 新增 `max_chars=1000` 参数，超长段落触发句级二次切分
- `clean_pdf_text()` 新增 `verbose=True` 诊断模式：打印被删行的分类统计和前 20 条样本，便于人工确认是否误删

**3.希望达到的目的**
- 所有 chunk 长度 ≤ 1000 字符，不出现超长 chunk
- 通过诊断输出确认 `clean_pdf_text` 只删噪声、不误删正文

In [ ]:
# 解析pdf需要安装依赖pymupdf，论文 PDF 和比赛规则 PDF 模块共用此依赖
# ⚠️⚠️⚠️ 
# 当前用的是pymupdf，优点是速度快，缺点是pymupdf 按坐标顺序提取，
# 会把左栏和右栏的文本混在一起，出现"左栏第一行→右栏第一行→左栏第二行"这种乱序
# ⚠️⚠️⚠️
!pip install pymupdf -q

zsh:1: command not found: pip


In [ ]:
# 调用pymupdf
import fitz  

# 通过colab上传
from google.colab import files as colab_files

def extract_pdf_text(pdf_path):
    # 逐页提取纯文本，页间用双换行拼接，形成段落切分的天然边界
    doc = fitz.open(pdf_path)
    pages_text = [page.get_text() for page in doc]
    doc.close()
    return '\n\n'.join(pages_text)


# ─── clean_pdf_text ───────────────────────────────────────────────────────
# 问题：「< 8 字符的行一律丢弃」规则可能误删公式行、短节标题
#       上一版只输出清洗前后字符数，无法确认被删的内容是否是噪声
# 解决：新增 verbose=True 参数，打印被删行的分类统计和前 20 条样本，
#       可视化确认过滤规则只删页眉页脚、不误删正文
# 目的：在修改过滤参数前先诊断，防止盲目调参误伤正文
def clean_pdf_text(text, verbose=False):
    lines = text.split('\n')
    cleaned = []
    removed_short = []      # < 8 字符的行
    removed_page_num = []   # 孤立页码
    in_references = False
    removed_ref_count = 0
    for line in lines:
        stripped = line.strip()
        if re.match(r'^(References|参考文献|REFERENCES)\s*$', stripped):
            in_references = True
        if in_references:
            removed_ref_count += 1
            continue
        if re.match(r'^\d{1,4}$', stripped):  # 孤立页码
            removed_page_num.append(repr(stripped))
            continue
        if 0 < len(stripped) < 8:  # 过短的页眉页脚，但保留空行
            removed_short.append(repr(stripped))
            continue
        cleaned.append(line)
    if verbose:
        print('[clean_pdf_text 诊断]')
        print(f'  原始行数: {len(lines)}')
        print(f'  参考文献段删除: {removed_ref_count} 行')
        print(f'  孤立页码删除:   {len(removed_page_num)} 行，样本: {removed_page_num[:10]}')
        print(f'  过短行删除:     {len(removed_short)} 行，样本（前20）：')
        for s in removed_short[:20]:
            print(f'    {s}')
        print(f'  输出行数: {len(cleaned)}')
    return '\n'.join(cleaned)


# ── 上传论文 PDF ──────────────────────────────────────────────────────────
# 建议上传：机器人运动控制相关论文 PDF（如 legged robot、locomotion、reinforcement learning 等主题）
# 也可以上传任意 PDF 进行测试
print('请上传论文 PDF 文件（点击下方按钮选择文件）：')
uploaded = colab_files.upload()
PAPER_PDF = list(uploaded.keys())[0]  # 取第一个上传的文件名

paper_raw = extract_pdf_text(PAPER_PDF)
# verbose=True：开启诊断，打印被删行样本，确认不误删正文
paper_text = clean_pdf_text(paper_raw, verbose=True)

print(f'\n原始提取: {len(paper_raw):,} 字符')
print(f'清洗后:   {len(paper_text):,} 字符（清除噪声 {len(paper_raw)-len(paper_text):,} 字符）')
print('\n前 300 字（确认是正常论文正文，对照诊断输出判断是否误删）：')
print(paper_text[:300])


In [ ]:
# 切分：doc_type='paper' 标记这是论文类文档
paper_chunks = chunk_text(paper_text, doc_id='paper_agile_locomotion_2020', doc_type='paper')
paper_lens = [c['char_len'] for c in paper_chunks]
print(f'切分出 {len(paper_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(paper_lens)} / 最大 {max(paper_lens)} / 平均 {sum(paper_lens)//len(paper_lens)}')
print('\n示例 chunk[2]（跳过前两个，通常是标题/作者）：')
print(paper_chunks[2]['chunk_text'][:300])

In [ ]:
# 还原校验：pass=True 才能继续往下做 Embedding
paper_check = validate_chunks(paper_text, paper_chunks)
print(f'[论文 PDF] pass={paper_check["pass"]}, 总 chunk={paper_check["total_chunks"]}, mismatch={paper_check["mismatch_count"]}')
if paper_check['mismatches']:
    print('有问题的 chunk：', paper_check['mismatches'])

---
## 模块 2：规则 PDF

### 针对的问题
- 规则文档为分条列举风格，段落较短，但仍可能出现超长段落
- 与模块 1 共用 `clean_pdf_text()`，同样需要验证诊断输出

### 解决方式
- 复用模块 1 的提取 + 清洗函数，开启 `verbose=True` 诊断
- 切分使用同一套 `chunk_text()`，带句级二次切分兜底

### 希望达到的目的
- 与论文 PDF 对比 chunk 长度分布，规则文档预期平均长度更短
- 验证同一套切分参数对不同风格文档的适应性

### 依赖
- pymupdf：已在模块 1 安装，无需重复


In [ ]:
# 上传规则 / 技术手册类 PDF
# 建议上传：比赛规则、技术规范、硬件手册等结构化文档
# 也可以上传任意 PDF 进行测试
print('请上传规则 PDF 文件（点击下方按钮选择文件）：')
uploaded_rule = colab_files.upload()
RULE_PDF = list(uploaded_rule.keys())[0]  # 取第一个上传的文件名

rule_raw = extract_pdf_text(RULE_PDF)
# verbose=True：与模块1保持一致，诊断被删行，对比不同风格文档的过滤结果
rule_text = clean_pdf_text(rule_raw, verbose=True)

print(f'\n原始提取: {len(rule_raw):,} 字符')
print(f'清洗后:   {len(rule_text):,} 字符（清除噪声 {len(rule_raw)-len(rule_text):,} 字符）')
print('\n前 300 字：')
print(rule_text[:300])


In [ ]:
# 切分
rule_chunks = chunk_text(rule_text, doc_id='rule_robocon_2026', doc_type='rule')
rule_lens = [c['char_len'] for c in rule_chunks]
print(f'切分出 {len(rule_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(rule_lens)} / 最大 {max(rule_lens)} / 平均 {sum(rule_lens)//len(rule_lens)}')
print('\n示例 chunk[0]：')
print(rule_chunks[0]['chunk_text'][:300])

In [ ]:
# 还原校验
rule_check = validate_chunks(rule_text, rule_chunks)
print(f'[规则 PDF] pass={rule_check["pass"]}, 总 chunk={rule_check["total_chunks"]}, mismatch={rule_check["mismatch_count"]}')
if rule_check['mismatches']:
    print('有问题的 chunk：', rule_check['mismatches'])

---
## 模块 3：arXiv 论文

### 针对的问题
- arXiv 论文与本地论文 PDF 同为学术文本，存在相同的超长段落问题
- 下载路径之前写死为 `实验样本/` 目录，Colab 环境中该目录不存在，导致报错

### 解决方式
- 下载路径改为当前目录（去掉 `实验样本/` 前缀）
- 复用 `chunk_text()` 的句级二次切分逻辑

### 希望达到的目的
- 验证 arXiv API 自动获取 + 切分的完整流程可用
- chunk 长度受控，不出现超长截断

### 依赖
- arxiv：`pip install arxiv`
- requests：下载 PDF 文件


In [ ]:
# arxiv：封装了 arXiv API 的调用，免去手动拼接 URL 和解析 XML
# requests：下载论文 PDF 文件
!pip install arxiv requests -q

In [ ]:
import arxiv
import requests
import os

# 通过 arXiv API 搜索论文，取第一条结果的元信息
client = arxiv.Client()
search = arxiv.Search(
    query='legged robot locomotion reinforcement learning',
    max_results=1,
    sort_by=arxiv.SortCriterion.Relevance
)
result = next(client.results(search))

# 元信息将来存入数据库 documents 表，用于来源展示
arxiv_meta = {
    'arxiv_id': result.entry_id.split('/')[-1],
    'title': result.title,
    'authors': [a.name for a in result.authors],
    'published': str(result.published.date()),
    'categories': result.categories,
    'pdf_url': result.pdf_url,
}
print('元信息：')
for k, v in arxiv_meta.items():
    print(f'  {k}: {v}')

In [ ]:
# 下载 PDF 并提取文本，复用模块 1 的函数
arxiv_pdf_path = f'arxiv_{arxiv_meta["arxiv_id"]}.pdf'
if not os.path.exists(arxiv_pdf_path):
    # 文件不存在才下载，避免重复下载
    r = requests.get(arxiv_meta['pdf_url'], timeout=60)
    with open(arxiv_pdf_path, 'wb') as f:
        f.write(r.content)
    print(f'下载完成：{len(r.content):,} 字节')
else:
    print(f'文件已存在，跳过下载')

arxiv_raw = extract_pdf_text(arxiv_pdf_path)
arxiv_text = clean_pdf_text(arxiv_raw)
print(f'提取: {len(arxiv_raw):,} 字符 → 清洗后: {len(arxiv_text):,} 字符')

In [ ]:
# 切分，doc_id 用 arxiv_id 确保唯一性且可追溯
arxiv_chunks = chunk_text(arxiv_text, doc_id=f'arxiv_{arxiv_meta["arxiv_id"]}', doc_type='arxiv')
arxiv_lens = [c['char_len'] for c in arxiv_chunks]
print(f'切分出 {len(arxiv_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(arxiv_lens)} / 最大 {max(arxiv_lens)} / 平均 {sum(arxiv_lens)//len(arxiv_lens)}')

In [ ]:
# 还原校验
arxiv_check = validate_chunks(arxiv_text, arxiv_chunks)
print(f'[arXiv] pass={arxiv_check["pass"]}, 总 chunk={arxiv_check["total_chunks"]}, mismatch={arxiv_check["mismatch_count"]}')
if arxiv_check['mismatches']:
    print('有问题的 chunk：', arxiv_check['mismatches'])

---
## 模块 4：GitHub 项目

### 针对的问题
- 之前样本固定为 SpotMicroESP32，难以更换，且出现极短 chunk（最小 18 字符）——空标题被切成独立 chunk
- 样本质量和数量不可控

### 解决方式
- 改为用户提供 GitHub 仓库链接（如 `https://github.com/owner/repo`）
- 代码从链接中自动解析 owner/repo，通过 GitHub API 递归获取所有 .md 文件内容
- `chunk_markdown()` 新增空标题过滤：标题行以外正文 < min_chars 时整段跳过

### 希望达到的目的
- 用户可随时更换为任意 GitHub 仓库，灵活测试不同样本
- 消除极短 chunk，GitHub 来源最小 chunk ≥ 80 字符

### 依赖
- requests：已安装


In [ ]:
import time
import re as _re

def parse_github_url(url):
    """从 GitHub 仓库 URL 解析 owner 和 repo。"""
    m = _re.search(r'github\.com/([^/]+)/([^/]+)', url)
    if not m:
        raise ValueError(f'无法从链接解析 owner/repo：{url}')
    return m.group(1), m.group(2).rstrip('/')

def fetch_git_md_files(owner, repo, platform='github', path=''):
    """递归遍历 GitHub 或 Gitee 仓库，返回所有 .md 文件的路径和下载地址。"""
    if platform == 'github':
        api_url = f'https://api.github.com/repos/{owner}/{repo}/contents/{path}'
    else:
        api_url = f'https://gitee.com/api/v5/repos/{owner}/{repo}/contents/{path}'
    resp = requests.get(api_url, timeout=30)
    resp.raise_for_status()
    items = resp.json()
    md_files = []
    for item in items:
        if item['type'] == 'file' and item['name'].lower().endswith('.md'):
            if platform == 'github':
                download_url = item['download_url']
            else:
                download_url = f'https://gitee.com/{owner}/{repo}/raw/master/{item["path"]}'
            md_files.append({'path': item['path'], 'download_url': download_url})
        elif item['type'] == 'dir':
            time.sleep(0.3)
            md_files.extend(fetch_git_md_files(owner, repo, platform, item['path']))
    return md_files


# 用户提供 GitHub 仓库链接，代码自动解析并获取所有 .md 文件
# 示例：https://github.com/michaelkubina/SpotMicroESP32
GITHUB_URL = input('请输入 GitHub 仓库链接：').strip()
github_owner, github_repo = parse_github_url(GITHUB_URL)
print(f'解析结果：owner={github_owner}, repo={github_repo}')
print('正在递归遍历仓库...')
github_md_files = fetch_git_md_files(github_owner, github_repo, platform='github')
print(f'找到 {len(github_md_files)} 个 .md 文件：')
for f in github_md_files:
    print(f'  {f["path"]}')


In [ ]:
# 逐个拉取 .md 文件内容，切分并汇总
github_all_chunks = []
github_texts = {}

for md_file in github_md_files:
    resp = requests.get(md_file['download_url'], timeout=30)
    md_text = resp.text
    github_texts[md_file['path']] = md_text
    if not md_text.strip():
        continue
    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'github_{github_repo}_{safe_path}'
    chunks = chunk_markdown(md_text, doc_id=doc_id, doc_type='github')
    github_all_chunks.extend(chunks)
    print(f'  {md_file["path"]:45s} -> {len(chunks):3d} chunks')

print(f'\nGitHub 合计：{len(github_all_chunks)} 个 chunk，来自 {len(github_md_files)} 个文件')


In [ ]:
# 还原校验：按文件逐个校验
github_mismatch_total = 0
for md_file in github_md_files:
    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'github_{github_repo}_{safe_path}'
    file_chunks = [c for c in github_all_chunks if c['doc_id'] == doc_id]
    if not file_chunks:
        continue
    check = validate_chunks(github_texts[md_file['path']], file_chunks)
    github_mismatch_total += check['mismatch_count']
    if check['mismatch_count'] > 0:
        print(f'  FAIL {md_file["path"]}: {check["mismatch_count"]} 个 mismatch')

github_lens = [c['char_len'] for c in github_all_chunks]
print(f'[GitHub] 总 mismatch={github_mismatch_total}, pass={github_mismatch_total == 0}')
if github_lens:
    print(f'[GitHub] chunk 长度：最小 {min(github_lens)} / 最大 {max(github_lens)} / 平均 {sum(github_lens)//len(github_lens)}')


---
## 模块 5：Gitee 项目

### 针对的问题
- 之前样本固定为 tinymal-spirit40-t，仅有 1 个 README、1 个 chunk，不足以验证模块泛化能力
- 与 GitHub 模块同样存在极短 chunk 问题

### 解决方式
- 改为用户提供 Gitee 仓库链接（如 `https://gitee.com/owner/repo`）
- 代码从链接中自动解析 owner/repo，通过 Gitee API 递归获取所有 .md 文件内容
- 复用 `fetch_git_md_files()` 函数（已在模块 4 定义），传 `platform='gitee'` 即可

### 希望达到的目的
- 补充有效的 Gitee 样本，chunk 数量足够反映真实分布
- 验证 Markdown 切分对中文内容的效果

### 依赖
- requests：已安装
- `fetch_git_md_files`：已在模块 4 定义


In [ ]:
import re as _re

def parse_gitee_url(url):
    """从 Gitee 仓库 URL 解析 owner 和 repo。"""
    m = _re.search(r'gitee\.com/([^/]+)/([^/]+)', url)
    if not m:
        raise ValueError(f'无法从链接解析 owner/repo：{url}')
    return m.group(1), m.group(2).rstrip('/')


# 用户提供 Gitee 仓库链接，代码自动解析并获取所有 .md 文件
# 示例：https://gitee.com/tinymal/tinymal-spirit40-t
GITEE_URL = input('请输入 Gitee 仓库链接：').strip()
gitee_owner, gitee_repo = parse_gitee_url(GITEE_URL)
print(f'解析结果：owner={gitee_owner}, repo={gitee_repo}')
print('正在递归遍历仓库...')
gitee_md_files = fetch_git_md_files(gitee_owner, gitee_repo, platform='gitee')
print(f'找到 {len(gitee_md_files)} 个 .md 文件：')
for f in gitee_md_files:
    print(f'  {f["path"]}')


In [ ]:
# 逐个拉取 .md 文件内容，切分并汇总
gitee_all_chunks = []
gitee_texts = {}

for md_file in gitee_md_files:
    resp = requests.get(md_file['download_url'], timeout=30)
    md_text = resp.text
    gitee_texts[md_file['path']] = md_text
    if not md_text.strip():
        continue
    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'gitee_{gitee_repo}_{safe_path}'
    chunks = chunk_markdown(md_text, doc_id=doc_id, doc_type='gitee')
    gitee_all_chunks.extend(chunks)
    print(f'  {md_file["path"]:45s} -> {len(chunks):3d} chunks')

print(f'\nGitee 合计：{len(gitee_all_chunks)} 个 chunk，来自 {len(gitee_md_files)} 个文件')


In [ ]:
# 还原校验
gitee_mismatch_total = 0
for md_file in gitee_md_files:
    safe_path = md_file['path'].replace('/', '_').replace('.', '_').lower()
    doc_id = f'gitee_{gitee_repo}_{safe_path}'
    file_chunks = [c for c in gitee_all_chunks if c['doc_id'] == doc_id]
    if not file_chunks:
        continue
    check = validate_chunks(gitee_texts[md_file['path']], file_chunks)
    gitee_mismatch_total += check['mismatch_count']
    if check['mismatch_count'] > 0:
        print(f'  FAIL {md_file["path"]}: {check["mismatch_count"]} 个 mismatch')

print(f'[Gitee] 总 mismatch={gitee_mismatch_total}, pass={gitee_mismatch_total == 0}')
if gitee_all_chunks:
    gitee_lens = [c['char_len'] for c in gitee_all_chunks]
    print(f'[Gitee] chunk 长度：最小 {min(gitee_lens)} / 最大 {max(gitee_lens)} / 平均 {sum(gitee_lens)//len(gitee_lens)}')


---
## 模块 6：普通网页

### 针对的问题
- 之前样本固定为 ROS 2 官网，该网站部署了 Anubis PoW 反爬保护，脚本直接抓取得到的是反爬提示而非正文，内容无效

### 解决方式
- 改为用户提供 URL，代码自动抓取
- 建议选择无反爬保护的页面（如 Wikipedia、ROS Wiki、官方文档镜像站等）
- doc_id 改为从 URL 动态生成，不再写死

### 希望达到的目的
- 获取真实有效的网页正文内容（不是反爬提示）
- 验证 HTML 解析 + 切分流程对真实网页的效果

### 推荐 URL（无反爬）
- `https://wiki.ros.org/ROS/Tutorials`
- `https://en.wikipedia.org/wiki/Robot_locomotion`
- `https://en.wikipedia.org/wiki/Legged_robot`

### 依赖
- beautifulsoup4 + lxml：解析 HTML


In [ ]:
# beautifulsoup4：解析 HTML，提取指定标签内的文字
# lxml：beautifulsoup4 的解析器后端，比默认 html.parser 更快更准确
!pip install beautifulsoup4 lxml -q

In [ ]:
from bs4 import BeautifulSoup

def extract_web_text(url):
    """下载 HTML，优先取 main 标签正文，去掉导航栏等噪声。"""
    headers = {'User-Agent': 'Mozilla/5.0'}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    soup = BeautifulSoup(resp.text, 'lxml')
    for tag in soup(['script', 'style', 'nav', 'footer', 'header']):
        tag.decompose()
    content = soup.find('main') or soup.find('article') or soup.find('body')
    if content is None:
        return ''
    paragraphs = []
    for elem in content.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'li', 'pre']):
        text = elem.get_text(separator=' ', strip=True)
        if text:
            paragraphs.append(text)
    return '\n\n'.join(paragraphs)


# 用户提供 URL，代码自动抓取正文
# 建议选择无反爬保护的页面，例如：
#   https://wiki.ros.org/ROS/Tutorials
#   https://en.wikipedia.org/wiki/Robot_locomotion
WEB_URL = input('请输入网页 URL：').strip()
web_text = extract_web_text(WEB_URL)

print(f'URL：{WEB_URL}')
print(f'提取文本长度: {len(web_text):,} 字符')
print('\n前 400 字（确认是页面正文，不是导航栏或反爬提示）：')
print(web_text[:400])


In [ ]:
# 切分，doc_id 从 URL 生成，确保唯一性且可追溯
import re as _re
web_doc_id = 'web_' + _re.sub(r'[^a-z0-9]', '_', WEB_URL.lower())[:60]
web_chunks = chunk_text(web_text, doc_id=web_doc_id, doc_type='web')
web_lens = [c['char_len'] for c in web_chunks]
print(f'切分出 {len(web_chunks)} 个 chunk')
print(f'chunk 长度：最小 {min(web_lens)} / 最大 {max(web_lens)} / 平均 {sum(web_lens)//len(web_lens)}')
print('\n示例 chunk[0]：')
print(web_chunks[0]['chunk_text'][:300])


In [ ]:
# 还原校验
web_check = validate_chunks(web_text, web_chunks)
print(f'[网页] pass={web_check["pass"]}, 总 chunk={web_check["total_chunks"]}, mismatch={web_check["mismatch_count"]}')
if web_check['mismatches']:
    print('有问题的 chunk：', web_check['mismatches'])

---
## 汇总：跨来源 chunk 长度分布对比

对比六类来源的 chunk 平均长度。

**为什么要对比：** 不同来源文档风格不同，chunk 长度差异大时（超过 2 倍），向量检索排序会有偏差——长 chunk 包含更多词汇，在相似度计算上天然占优。如果差异过大，后续 Re-ranking 阶段需要引入长度归一化。

**注意：** 当前只有六个样本，不能下结论，用于建立初步直觉。接入更多真实论文后（10-15 篇），回来重跑这个单元格。

In [ ]:
# 汇总各来源的 chunk 长度统计
summary = [
    ('论文 PDF',  paper_lens),
    ('规则 PDF',  rule_lens),
    ('arXiv',    arxiv_lens),
    ('GitHub',   github_lens),
    ('Gitee',    [c['char_len'] for c in gitee_all_chunks] if gitee_all_chunks else [0]),
    ('网页',     web_lens),
]

print(f'{"来源":<12} {"chunk数":>8} {"最小":>8} {"最大":>8} {"平均":>8}')
print('-' * 50)
avgs = []
for name, lens in summary:
    if not lens or max(lens) == 0:
        print(f'{name:<12} {"(无数据)":>8}')
        continue
    avg = sum(lens) // len(lens)
    avgs.append((name, avg))
    print(f'{name:<12} {len(lens):>8} {min(lens):>8} {max(lens):>8} {avg:>8}')

if len(avgs) >= 2:
    max_avg = max(a for _, a in avgs)
    min_avg = max(1, min(a for _, a in avgs))
    ratio = max_avg / min_avg
    print(f'\n最大/最小平均长度比值：{ratio:.1f}x')
    if ratio > 2:
        print('提示：差异较大，后续 Re-ranking 阶段需关注长度归一化问题')
    else:
        print('提示：差异在可接受范围内，暂不需要特殊处理')

In [ ]:
# 所有来源的还原校验汇总
print('=== 还原校验汇总 ===')
checks = [
    ('论文 PDF',  paper_check),
    ('规则 PDF',  rule_check),
    ('arXiv',    arxiv_check),
    ('网页',     web_check),
]
for name, check in checks:
    status = 'PASS' if check['pass'] else 'FAIL'
    print(f'  {name:<12} {status}  chunk={check["total_chunks"]}, mismatch={check["mismatch_count"]}')

# GitHub 和 Gitee 汇总
print(f'  {"GitHub":<12} {"PASS" if github_mismatch_total == 0 else "FAIL"}  chunk={len(github_all_chunks)}, mismatch={github_mismatch_total}')
print(f'  {"Gitee":<12} {"PASS" if gitee_mismatch_total == 0 else "FAIL"}  chunk={len(gitee_all_chunks)}, mismatch={gitee_mismatch_total}')
print('\n全部 PASS 后可继续进行 Step 2：Embedding 向量化')